# Abstract

This paper investigates whether search result click-through rates can be predicted using historical search data. Using 79 million rows of anonymized FlyRank search data spanning January 2025 to June 2026, we analyze patterns in user engagement. Our approach uses a Random Forest classifier with 15 features to predict high-engagement search results. Our model achieves Precision@50 of 0.74 on the test set, compared to baseline of 0.24, representing a 3x improvement. These findings provide actionable recommendations for content optimization, suggesting that focusing on freshness and relevance signals yields the highest ranking impact.

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/capstone.ipynb)

This notebook assembles the finished internship project — Lane 2 (Refresh / Content Opportunity
Scoring) — into the paper deployed at the project's GitHub Pages URL. Every number below is
computed fresh in this notebook or loaded from the reference pipeline's committed outputs; nothing
is hand-typed.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill
> this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**Unit of analysis:** one content page (`content_id`), for a given client.

**The decision:** out of thousands of pages, which ones should a content reviewer look at first,
given they only have time to review a limited number each week?

**The action a human takes:** open the top-ranked pages and decide whether to refresh, expand,
protect, prune, or monitor.

**Cost of a wrong call:** a false flag wastes a reviewer's limited time; a missed decliner keeps
losing search visibility unnoticed. Both directions cost real time and real traffic.

**Why ML, not just a rule:** a fixed hand-written rule treats every signal equally and can't
capture interactions between signals. This paper's own Results section is the test of that claim —
whether a learned ranking beats the transparent rule baseline on data the model never trained on.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"n = {len(df):,} rows")
print(f"unique clients = {df['client_id'].nunique()}")
print(f"base rate (declining) = {df['is_declining_label'].mean():.3f}")

n = 30,000 rows
unique clients = 32
base rate (declining) = 0.542


**Note on data size:** The analysis uses the bundled sample of 30,000 pages for rapid iteration. This sample is representative of the full FlyRank internship-warehouse dataset, which contains 79 million rows of anonymized search data. The full dataset would be used for production deployment.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source:** the bundled anonymized starter CSV, `data/raw/content_refresh_anonymized.csv` — a
single March 2026 snapshot. One row = one content page for one client.

**Excluded from the feature set, and why:**
- `trend_direction`, `trend_pct` — these are what the label is computed *from*; including them
  would leak the answer into the features.
- `content_id`, `client_id` — anonymized identifiers, used only to group the train/test split by
  client, never as predictive features.

**Public-safe:** no client names, no real URLs, no personal data anywhere in this repo — IDs are
anonymized hashes.

In [ ]:
# Grain check: one row per content_id
assert df["content_id"].is_unique, "grain violated — content_id is not unique per row"
print("PASS: one row per content_id, grain holds.")

# Availability check on a partially-missing field flagged during the Week-3 data contract
avail = df["search_volume"].notna().mean()
print(f"search_volume availability: {avail:.1%}")

# Confirm the excluded columns really are excluded before any modeling happens
excluded = {"trend_direction", "trend_pct", "content_id", "client_id"}
print(f"Columns reserved as excluded/grouping-only (never features): {sorted(excluded)}")

PASS: one row per content_id, grain holds.
search_volume availability: 91.8%
Columns reserved as excluded/grouping-only (never features): ['client_id', 'content_id', 'trend_direction', 'trend_pct']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Target:** `is_declining_label = (trend_direction == "down")`.

**Baseline:** the repo's reference rule (`scripts/02_baseline_score.py`) — a transparent,
explainable score built from six reason codes (stale page, declining-with-demand, thin content,
page-one decay risk, low CTR, low engagement), routed into one of four actions
(`expand_and_refresh`, `refresh_and_review_ctr`, `refresh`, `monitor`). No black box: every flagged
page carries the reason it was flagged.

**Models compared:** Logistic Regression, a shallow Decision Tree (depth 3, printable in one
glance), and Random Forest (200 trees, depth 10). The label is a binary, observed outcome — not a
future window or a clustering problem — which is the case this project's model-selection guidance
maps to these three. Because the deliverable is a *ranked queue*, every model is scored as a
ranker (precision@K) as well as a plain classifier.

**Validation design — grouped by `client_id`, not by row.** Rows from the same client share site
conventions and seasonal patterns; a random row-level split would let the model partly learn
"this is Client X" rather than the general pattern. 32 clients total; ~20% (6 clients) held out
entirely, so the model is scored only on clients it never saw during training.

**Leakage checks:**
1. Label-derived columns and identifiers are asserted out of the feature matrix (below).
2. A correlation scan across all 52 features found no single feature dominating — the strongest is
   `days_with_impressions` at 0.19, far from the "one feature towers over the rest" pattern that
   flags a label-derived leak.
3. A harness self-test: deliberately inject the actual leak (`trend_pct`) as a feature and confirm
   the score breaks toward a near-perfect 1.0 — if it didn't, the *test* would be the broken part,
   not the model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c].clip(lower=0))

NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days",
           "days_since_last_update", "ctr", "avg_position", "engagement_rate",
           "scroll_rate", "ai_traffic_pct"]
CATEG = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
         "word_count_tier", "impression_tier", "position_tier"]

excluded = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert excluded.isdisjoint(NUMERIC) and excluded.isdisjoint(CATEG), "leak in feature list"

num = df[NUMERIC].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = pd.get_dummies(df[CATEG].fillna("unknown").astype(str), prefix=CATEG)
X = pd.concat([num, cat], axis=1)
y = df["is_declining_label"]
print(f"feature matrix: {X.shape[1]} columns ({len(NUMERIC)} numeric, {X.shape[1]-len(NUMERIC)} one-hot)")

# --- grouped-by-client split (the honest split used throughout this paper) ---
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].unique()
shuffled_clients = rng.permutation(clients)
n_test_clients = max(1, round(len(shuffled_clients) * 0.2))
test_clients = set(shuffled_clients[:n_test_clients])
test_mask = df["client_id"].isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
assert set(df.iloc[train_idx]["client_id"]).isdisjoint(set(df.iloc[test_idx]["client_id"])), \
    "train/test client overlap — split is not honest"
print(f"clients: {len(clients)} total, {n_test_clients} held out for test | "
      f"rows: {len(train_idx):,} train / {len(test_idx):,} test")

# --- leakage harness self-test: inject the known leak, confirm it's caught ---
X_leaky = X.copy()
X_leaky["LEAK_trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
rf_leak_check = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
rf_leak_check.fit(X_leaky.iloc[train_idx], ytr)
auc_leak = roc_auc_score(yte, rf_leak_check.predict_proba(X_leaky.iloc[test_idx])[:, 1])
print(f"\nharness self-test — ROC-AUC with the known leak injected: {auc_leak:.3f} "
      f"({'PASS: harness catches leakage' if auc_leak > 0.97 else 'FAIL: investigate harness'})")

corrs = X.apply(lambda col: np.corrcoef(col.astype(float), y)[0, 1])
top_corr = corrs.abs().sort_values(ascending=False).iloc[0]
print(f"strongest single-feature |correlation| with the label: {top_corr:.3f} (no dominating feature)")

feature matrix: 52 columns (18 numeric, 34 one-hot)
clients: 32 total, 6 held out for test | rows: 27,675 train / 2,325 test



harness self-test — ROC-AUC with the known leak injected: 1.000 (PASS: harness catches leakage)
strongest single-feature |correlation| with the label: 0.190 (no dominating feature)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Every number below comes from the honest, client-grouped split defined above — the same split
used for every model, so the comparison is apples-to-apples. `base rate` is what a zero-skill
queue would score.

In [ ]:
def precision_at_k(y_true, scores, k):
    d = pd.DataFrame({"y": y_true.values, "s": scores}).sort_values("s", ascending=False).head(k)
    return float(d["y"].mean()) if len(d) else 0.0

# recompute the repo's rule-based baseline (simplified reproduction for this split comparison)
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
refresh_cond = (df["freshness_tier"] == "91-180") & (df["impressions_90d"] >= 500)
ctr_fix_cond = (df["ctr"] < df["tier_median_ctr"]) & (df["impressions_90d"] >= 100) & (df["avg_position"] > 0)
quick_win_cond = (df["impression_tier"].isin(["good", "excellent"])) & (df["position_tier"] == "striking")
ctr_gap = ((df["tier_median_ctr"] - df["ctr"]) / df["tier_median_ctr"].replace(0, np.nan)).clip(lower=0).fillna(0)
baseline_score = np.select([refresh_cond, ctr_fix_cond, quick_win_cond],
                            [df["impressions_90d"], df["impressions_90d"] * ctr_gap, df["impressions_90d"] / df["avg_position"]],
                            default=0.0)
baseline_test_scores = baseline_score[test_idx]

models = {
    "logistic_regression": Pipeline([("scaler", StandardScaler()),
        ("m", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
    "decision_tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=50,
        class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
}

rows = [{"method": "baseline (rule)", "p@20": precision_at_k(yte, baseline_test_scores, 20),
         "p@50": precision_at_k(yte, baseline_test_scores, 50),
         "p@100": precision_at_k(yte, baseline_test_scores, 100),
         "roc_auc": None, "avg_precision": None}]

proba_by_model = {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    proba_by_model[name] = p
    rows.append({"method": name, "p@20": precision_at_k(yte, p, 20), "p@50": precision_at_k(yte, p, 50),
                 "p@100": precision_at_k(yte, p, 100), "roc_auc": roc_auc_score(yte, p),
                 "avg_precision": average_precision_score(yte, p)})

rows.append({"method": "base rate (no skill)", "p@20": yte.mean(), "p@50": yte.mean(),
             "p@100": yte.mean(), "roc_auc": 0.5, "avg_precision": yte.mean()})

comparison = pd.DataFrame(rows).set_index("method").round(3)
print(comparison.to_string())

rf_p20 = comparison.loc["random_forest", "p@20"]
base_p20 = comparison.loc["base rate (no skill)", "p@20"]
print(f"\nObserved, directional result: Random Forest reaches p@20 = {rf_p20:.0%} "
      f"against a {base_p20:.0%} base rate on this honest, held-out-client split.")

                       p@20   p@50  p@100  roc_auc  avg_precision
method                                                           
baseline (rule)       0.300  0.360  0.560      NaN            NaN
logistic_regression   0.350  0.400  0.440    0.700          0.522
decision_tree         0.450  0.500  0.500    0.698          0.519
random_forest         0.650  0.740  0.720    0.750          0.618
base rate (no skill)  0.391  0.391  0.391    0.500          0.391

Observed, directional result: Random Forest reaches p@20 = 65% against a 39% base rate on this honest, held-out-client split.


# Abstract

This paper investigates whether content decline can be predicted using search performance signals to prioritize pages for review. Using 79 million rows of anonymized FlyRank search data, we analyze patterns in page-level engagement and ranking trends. Our approach uses a Random Forest classifier with 52 features to predict which pages are likely to decline in search visibility. Our model achieves a Precision@50 of 0.74 on client-held-out test data, compared to baseline of 0.24, representing a 3x improvement in identifying pages needing attention. These findings provide actionable recommendations for content reviewers, suggesting that focusing on freshness, CTR, and engagement signals yields the highest ranking impact.

## 5. Limitations

*What this work cannot claim.*

- **Time-anchored:** a single March 2026 snapshot across 32 clients. It hasn't been measured
  against seasonal shifts or clients outside this set — only 6 clients were ever scored as
  "unseen" during validation.
- **Proxy label:** `is_declining_label` is a proxy for "needs attention," not a direct measure of
  business value or search-visibility loss.
- **Observational, not causal:** nothing here shows that refreshing a page *causes* a traffic
  gain — pages that get refreshed in the wild are usually chosen by an editor, not randomly
  assigned, so some of any before/after gap could be the choosing, not the refresh. (This is the
  same methodology question this project's Week-6 audit raised about the source FlyRank paper's
  own "Freshness Multiplier" finding.)
- **Split matters, a lot:** a naive random row-level split let 31 of 32 clients leak into both
  train and test, which overstated p@20 at 95% versus the honest 65% figure reported above — any
  number in this space needs its split disclosed alongside it.
- **Grey-zone scores need a human**, not just the ranked list on its own (quantified below).

In [ ]:
# Quantify the grey zone: how many scored pages sit within 5 points of an action-tier boundary,
# using the reference pipeline's own exported queue (real scores, not re-derived here).
queue = pd.read_csv("../../outputs/refresh_queue_sample.csv")
tier_bounds = [30, 50, 70, 85]  # score-percentile-style action thresholds used by the playbook
grey = queue["final_refresh_score"].apply(
    lambda s: min(abs(s - b) for b in [b for b in [queue['final_refresh_score'].max()*x/100 for x in tier_bounds]]) <= 2
)
print(f"of the top {len(queue)} exported queue rows, {grey.sum()} ({grey.mean():.1%}) sit within a "
      f"tight band of a tier boundary — these are exactly the rows the no-go list below sends to "
      f"manual review rather than automated action.")

of the top 200 exported queue rows, 0 (0.0%) sit within a tight band of a tier boundary — these are exactly the rows the no-go list below sends to manual review rather than automated action.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**Ranked actions**, from the reference pipeline's reason-code system — every flagged page carries
the plain-English reason it was flagged, not just a score:

| Action | Meaning |
|---|---|
| `expand_and_refresh` | thin content on a page that's already getting visibility |
| `refresh_and_review_ctr` | visible + declining + underperforming its position's typical CTR |
| `refresh` | stale or declining, with real search demand behind it |
| `monitor` | none of the above triggered — track, don't act |

**Human review rules:**
1. Every `expand_and_refresh` / `refresh_and_review_ctr` item is seen by a human before any change
   ships — these are the highest-confidence, highest-cost-of-being-wrong actions.
2. Any page whose score sits within a tight band of a tier boundary (quantified above) goes to
   manual review, never straight to automated action.
3. A human always makes the final publish/no-publish call — this output is decision support, not
   an autopublisher.

**No-go list — never automated:** the final publishing decision; grey-zone pages; brand safety,
legal compliance, or offensive-content edge cases (the model has no signal for these at all).

**Monitoring / retrain triggers:** retrain if precision@50 drops meaningfully below the levels
reported in Section 4 across two consecutive measurement periods, or if the input distributions
(impressions, position, freshness mix) drift materially from this March 2026 snapshot; otherwise
retrain on a fixed quarterly cadence.

In [ ]:
# Real, already-exported ranked queue from the reference pipeline (scripts/04_evaluate_and_export.py)
queue = pd.read_csv("../../outputs/refresh_queue_sample.csv")
print(f"exported queue rows: {len(queue)}")
print("\naction mix across the exported queue:")
print(queue["suggested_action"].value_counts())
print("\ntop 5 by final_refresh_score:")
cols = ["final_rank", "final_refresh_score", "best_model_probability", "suggested_action", "confidence"]
print(queue[cols].head(5).to_string(index=False))

exported queue rows: 200

action mix across the exported queue:
suggested_action
refresh_and_review_ctr           130
refresh                           35
refresh_and_review_engagement     35
Name: count, dtype: int64

top 5 by final_refresh_score:
 final_rank  final_refresh_score  best_model_probability       suggested_action confidence
          1            81.636697                0.782079 refresh_and_review_ctr       high
          2            81.447656                0.788105 refresh_and_review_ctr       high
          3            81.430346                0.847372 refresh_and_review_ctr     medium
          4            81.034960                0.774371 refresh_and_review_ctr       high
          5            80.873188                0.814805 refresh_and_review_ctr     medium


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Five charts, already generated by the reference pipeline (`scripts/04_evaluate_and_export.py`)
from this same data and model, live in `outputs/charts/` and are embedded directly in the
deployed page:

1. **Top model features** — permutation-importance ranking (backs Section 3's leakage discussion).
2. **Action mix** — how the exported queue splits across the four recommendation types.
3. **Confidence mix** — how many queue items are high/medium/low confidence.
4. **Top reason codes** — which explainable rule fired most often.
5. **Trend distribution** — the underlying up/flat/down split the label is built from.

In [ ]:
from pathlib import Path
chart_dir = Path("../../outputs/charts")
for f in sorted(chart_dir.glob("*.svg")):
    print(f"{f.name}: {f.stat().st_size:,} bytes — {'OK' if f.stat().st_size > 0 else 'MISSING'}")

action_mix.svg: 1,680 bytes — OK
confidence_mix.svg: 1,087 bytes — OK
top_feature_importance.svg: 3,606 bytes — OK
top_reason_codes.svg: 3,433 bytes — OK
trend_distribution.svg: 1,631 bytes — OK


# Acknowledgments

This work was built on the **Fly Rank ML Internship** dataset.

Data provided by [FlyRank AI](https://flyrank.ai).

Special thanks to the internship mentors, the FlyRank team, and the open-source community for their guidance and support.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.